# AI Baseline Text-to-Research Notebook

Pipeline:

1. Load any text or Markdown source file.
2. Extract investment-relevant themes from the source.
3. Run AI Baseline retrieval against those themes.
4. Generate a grounded research note with an OpenRouter model.

Secrets are loaded from `.env` next to this notebook. Use `.env.example` as the template.

Start by editing and running the first code cell. Everything below it is the reusable workflow.

In [ ]:
# USER CONTROLS
# Keep real API keys in notebooks/.env. This cell only names the env vars to read.

import os
from pathlib import Path

INPUT_TEXT_PATH = "source_text.md"
OUTPUT_DIR_NAME = "ai_baseline_text_research_output"
RUN_LABEL = None  # None creates a UTC timestamped run folder.

OPENROUTER_API_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_LLM_MODEL = "~anthropic/claude-sonnet-latest"
AI_BASELINE_API_BASE_URL = "https://api-beta.ai-baseline.xyz/v2"

OPENROUTER_API_KEY_ENV = "OPENROUTER_API_KEY"
AI_BASELINE_API_KEY_ENV = "AI_BASELINE_API_KEY"


def _load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def _detect_notebook_root() -> Path:
    session_name = os.getenv("JPY_SESSION_NAME")
    if session_name:
        session_path = Path(session_name).expanduser()
        if session_path.suffix == ".ipynb":
            if not session_path.is_absolute():
                session_path = Path.cwd() / session_path
            return session_path.resolve().parent
    return Path.cwd().resolve()


NOTEBOOK_ROOT = _detect_notebook_root()
ENV_PATH = NOTEBOOK_ROOT / ".env"
_load_env_file(ENV_PATH)

_input_path = Path(INPUT_TEXT_PATH).expanduser()
if not _input_path.is_absolute():
    _input_path = NOTEBOOK_ROOT / _input_path

print("CONFIG")
print("------")
print(f"env file             : {ENV_PATH} ({'found' if ENV_PATH.exists() else 'not found'})")
print(f"input file           : {_input_path} ({'found' if _input_path.exists() else 'not found'})")
print(f"output dir           : {NOTEBOOK_ROOT / OUTPUT_DIR_NAME}")
print(f"run label            : {RUN_LABEL or 'timestamped'}")
print(f"openrouter api base  : {OPENROUTER_API_BASE_URL}")
print(f"openrouter model     : {OPENROUTER_LLM_MODEL}")
print(f"ai baseline api base : {AI_BASELINE_API_BASE_URL or 'not set'}")
print(f"openrouter key env   : {OPENROUTER_API_KEY_ENV} ({'loaded' if os.getenv(OPENROUTER_API_KEY_ENV) else 'missing'})")
print(f"ai baseline key env  : {AI_BASELINE_API_KEY_ENV} ({'loaded' if os.getenv(AI_BASELINE_API_KEY_ENV) else 'missing'})")

In [ ]:

import html
import importlib.util
import json
import os
import re
import sys
import textwrap
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from IPython.display import HTML, Markdown, display

In [ ]:
# Optional dependency bootstrap. This invokes the notebook `%pip` magic through IPython.

INSTALL_MISSING_NOTEBOOK_PACKAGES = True
NOTEBOOK_CORE_PACKAGE_IMPORTS = {
    "json_repair": "json-repair",
    "markdown": "markdown",
}
NOTEBOOK_RICH_PDF_PACKAGE_IMPORTS = {
    "weasyprint": "weasyprint",
    "reportlab": "reportlab",
}
NOTEBOOK_PACKAGE_IMPORTS = {
    **NOTEBOOK_CORE_PACKAGE_IMPORTS,
    **NOTEBOOK_RICH_PDF_PACKAGE_IMPORTS,
}


def missing_package_specs(package_imports: dict[str, str]) -> list[str]:
    return [
        package_name
        for import_name, package_name in package_imports.items()
        if importlib.util.find_spec(import_name) is None
    ]


def install_missing_notebook_packages(package_imports: dict[str, str], label: str) -> list[str]:
    missing_packages = missing_package_specs(package_imports)
    if not missing_packages:
        print(f"{label} package check passed.")
        return []
    if not INSTALL_MISSING_NOTEBOOK_PACKAGES:
        print(f"{label} package install disabled. Missing: {', '.join(missing_packages)}")
        return missing_packages

    ipython = globals().get("get_ipython", lambda: None)()
    if ipython is None:
        print(f"Install missing {label.lower()} packages with:")
        print(f"{sys.executable} -m pip install {' '.join(missing_packages)}")
        return missing_packages

    for import_name, package_name in package_imports.items():
        if importlib.util.find_spec(import_name) is not None:
            continue
        try:
            ipython.run_line_magic("pip", f"install {package_name}")
            importlib.invalidate_caches()
        except Exception as exc:
            print(f"Could not install {label.lower()} package: {package_name}")
            print(f"Install error: {exc}")
            continue
        if importlib.util.find_spec(import_name) is not None:
            print(f"Installed {label.lower()} package: {package_name}")
        else:
            print(
                f"Install command finished, but `{import_name}` is still not importable. "
                f"Check the `%pip install {package_name}` output above."
            )

    remaining_packages = missing_package_specs(package_imports)
    if remaining_packages:
        print(f"{label} package(s) still missing: " + ", ".join(remaining_packages))
    else:
        print(f"{label} package check passed after install.")
    return remaining_packages


remaining_core_packages = install_missing_notebook_packages(
    NOTEBOOK_CORE_PACKAGE_IMPORTS,
    "Core notebook",
)
remaining_pdf_packages = install_missing_notebook_packages(
    NOTEBOOK_RICH_PDF_PACKAGE_IMPORTS,
    "Rich PDF",
)
available_pdf_packages = [
    package_name
    for import_name, package_name in NOTEBOOK_RICH_PDF_PACKAGE_IMPORTS.items()
    if importlib.util.find_spec(import_name) is not None
]
if remaining_pdf_packages:
    print(
        "Rich PDF package(s) unavailable after bootstrap: "
        + ", ".join(remaining_pdf_packages)
        + ". PDF_RENDERER='auto' will try the remaining renderers and use plain fallback only as a last resort."
    )
if available_pdf_packages:
    print("Rich PDF renderer package(s) available: " + ", ".join(available_pdf_packages) + ".")
else:
    print("No rich PDF renderer package is available; PDF_RENDERER='auto' will use the plain fallback.")

In [ ]:
def load_dotenv(path: Path) -> None:
    if not path.exists():
        print(f'No env file found at {path}. Create it from .env.example before API calls.')
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def detect_notebook_root() -> Path:
    session_name = os.getenv('JPY_SESSION_NAME')
    if session_name:
        session_path = Path(session_name).expanduser()
        if session_path.suffix == '.ipynb':
            if not session_path.is_absolute():
                session_path = Path.cwd() / session_path
            return session_path.resolve().parent
    return Path.cwd().resolve()


# Keep the notebook self-contained: all relative paths resolve from this notebook's folder.
NOTEBOOK_ROOT = detect_notebook_root()
ENV_PATH = NOTEBOOK_ROOT / '.env'
load_dotenv(ENV_PATH)

OPENROUTER_API_BASE_URL = OPENROUTER_API_BASE_URL.rstrip('/')
AI_BASELINE_API_BASE_URL = AI_BASELINE_API_BASE_URL.rstrip('/')

# Financial research settings.
DOCUMENT_LABEL = "source text"
EVIDENCE_CORPUS_DESCRIPTION = "S&P 500 SEC filings"
AI_BASELINE_DOMAIN = "sandp_500"

OPENROUTER_TIMEOUT_SECONDS = 600
OPENROUTER_HTTP_REFERER = ""
OPENROUTER_APP_TITLE = "AI Baseline Financial Research Notebook"

AI_BASELINE_MODE = "agentic_research"
AI_BASELINE_EFFORT = "high"
AI_BASELINE_MAX_EVIDENCE = 800
AI_BASELINE_TIMEOUT_SECONDS = 300
AI_BASELINE_MAX_QUERY_CHARS = 4_000
AI_BASELINE_QUERY_CHAR_BUDGET = 3_800

# Topic extraction. Leave FOCUS_AREAS empty to let the model infer themes from the text.
MAX_TOPICS = 8
MIN_TOPICS = 3
FOCUS_AREAS: list[str] = []
EXTRACTION_CHUNK_CHARS = 12_000
# None means the notebook does not send an explicit output-token cap for JSON calls.
EXTRACTION_MAX_TOKENS: int | None = None
CONSOLIDATION_MAX_TOKENS: int | None = None
JSON_REPAIR_MAX_TOKENS: int | None = None

# Follow-up retrieval loop.
MAX_AI_BASELINE_TOPIC_QUERIES = MAX_TOPICS
ENABLE_AI_BASELINE_FOLLOWUPS = True
MAX_AI_BASELINE_FOLLOWUP_QUERIES = 2
AI_BASELINE_AUDIT_CONTEXT_CHARS = 30_000
AI_BASELINE_AUDIT_MAX_TOKENS = 2_500

# Final note controls.
ANSWER_TEMPERATURE = 0.2
ANSWER_MAX_TOKENS: int | None = None
ANSWER_COMPLETION_MAX_TOKENS: int | None = None
ANSWER_CONTEXT_MAX_CHARS = 180_000
ANSWER_CONTEXT_PER_SEARCH_MAX_CHARS = 45_000
ANSWER_CONTEXT_MAX_EVIDENCE_ENTRIES_PER_SEARCH = 35
ANSWER_CONTEXT_SOURCE_REGISTRY_MAX_CHARS = 10_000
REQUIRED_ANSWER_SECTION_HEADINGS = [
    'Executive View',
    'Source Themes and Evidence Status',
    'AI Baseline-Supported Findings',
    'Highest-Conviction Investable Insights',
    'Company and Ticker Implications',
    'Bull Case',
    'Bear Case and Key Risks',
    'What to Verify Next',
    'Watchlist / Action Framing',
]

# Display and export controls.
EXPORT_INVESTMENT_NOTE_PDF = True
PDF_RENDERER = "auto"  # auto tries WeasyPrint, ReportLab, then pandoc.
PDF_FILENAME = "investment_note.pdf"
REQUIRE_RICH_PDF_MARKDOWN = True
ALLOW_PLAIN_PDF_FALLBACK = False
PDF_MARKDOWN_EXTENSIONS = ['extra', 'sane_lists', 'tables']
SHOW_PROGRESS_BARS = True
SOURCE_PREVIEW_CHARS = 2_500
TOPICS_PREVIEW_CHARS = 5_000
AI_BASELINE_PREVIEW_CHARS = 5_000


def notebook_path(path: str | Path) -> Path:
    path = Path(path).expanduser()
    return path if path.is_absolute() else NOTEBOOK_ROOT / path


def ascii_progress_html(title: str, current: int, total: int, status: str = '') -> str:
    total = max(total, 1)
    current = min(max(current, 0), total)
    percent = current / total * 100
    width = 28
    filled = round(width * current / total)
    bar = '#' * filled + '-' * (width - filled)
    text = (
        f'{title}\n'
        f'[{bar}] {current:>2}/{total:<2} {percent:5.1f}%\n'
        f'{status}'
    )
    return f'''
    <pre style="background:#0b1020;color:#d6e2ff;border:1px solid #26324d;border-radius:6px;padding:12px 14px;margin:8px 0;font:12px/1.45 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;white-space:pre-wrap;">{html.escape(text)}</pre>
    '''


class NotebookProgress:
    def __init__(self, title: str, total: int, status: str = 'Starting') -> None:
        self.title = title
        self.total = max(total, 1)
        self.current = 0
        self.enabled = bool(globals().get('SHOW_PROGRESS_BARS', True))
        self.handle = None
        if self.enabled:
            self.handle = display(HTML(ascii_progress_html(title, self.current, self.total, status)), display_id=True)

    def update(self, current: int | None = None, status: str = '') -> None:
        self.current = self.current + 1 if current is None else current
        self.current = min(max(self.current, 0), self.total)
        if self.enabled and self.handle is not None:
            self.handle.update(HTML(ascii_progress_html(self.title, self.current, self.total, status)))

    def set_total(self, total: int, status: str = '') -> None:
        self.total = max(total, 1)
        self.current = min(max(self.current, 0), self.total)
        if status:
            self.update(self.current, status)

    def finish(self, status: str = 'Done') -> None:
        self.update(self.total, status)


def safe_run_name(label: str | None) -> str:
    if not label or not label.strip():
        return datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', label.strip()).strip('_.-')
    return safe or datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')


RUN_ID = safe_run_name(RUN_LABEL)
ARTIFACT_DIR = NOTEBOOK_ROOT / OUTPUT_DIR_NAME / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

input_source_path = notebook_path(INPUT_TEXT_PATH)


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Missing {name}. Add it to {ENV_PATH}.')
    return value


def post_json(url: str, headers: dict[str, str], payload: dict[str, Any], timeout_seconds: float) -> dict[str, Any]:
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={**headers, 'Content-Type': 'application/json'},
        method='POST',
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout_seconds) as response:
            raw_body = response.read().decode('utf-8')
    except urllib.error.HTTPError as exc:
        body = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'HTTP {exc.code} from {url}: {body[:1000]}') from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(f'Request failed for {url}: {exc.reason}') from exc
    decoded = json.loads(raw_body)
    if not isinstance(decoded, dict):
        raise ValueError(f'Expected JSON object from {url}')
    decoded.setdefault('_elapsed_seconds', time.perf_counter() - started)
    return decoded


def openrouter_headers() -> dict[str, str]:
    headers = {'Authorization': f'Bearer {require_env(OPENROUTER_API_KEY_ENV)}'}
    if OPENROUTER_HTTP_REFERER:
        headers['HTTP-Referer'] = OPENROUTER_HTTP_REFERER
    if OPENROUTER_APP_TITLE:
        headers['X-OpenRouter-Title'] = OPENROUTER_APP_TITLE
    return headers


def openrouter_chat(
    model: str,
    messages: list[dict[str, Any]],
    *,
    temperature: float = 0,
    max_tokens: int | None = None,
    response_format: dict[str, Any] | None = None,
) -> dict[str, Any]:
    payload: dict[str, Any] = {
        'model': model,
        'messages': messages,
        'temperature': temperature,
    }
    if max_tokens is not None:
        payload['max_tokens'] = max_tokens
    if response_format is not None:
        payload['response_format'] = response_format
    return post_json(
        f'{OPENROUTER_API_BASE_URL}/chat/completions',
        openrouter_headers(),
        payload,
        OPENROUTER_TIMEOUT_SECONDS,
    )


def openrouter_response_summary(response: dict[str, Any]) -> dict[str, Any]:
    choices = response.get('choices')
    choice = choices[0] if isinstance(choices, list) and choices and isinstance(choices[0], dict) else {}
    message = choice.get('message') if isinstance(choice, dict) else {}
    content = message.get('content') if isinstance(message, dict) else None
    return {
        'id': response.get('id'),
        'model': response.get('model'),
        'finish_reason': choice.get('finish_reason') if isinstance(choice, dict) else None,
        'native_finish_reason': choice.get('native_finish_reason') if isinstance(choice, dict) else None,
        'content_type': type(content).__name__,
        'content_length': len(content) if isinstance(content, str) else None,
        'usage': response.get('usage'),
    }


def openrouter_message_content(response: dict[str, Any]) -> str:
    choices = response.get('choices')
    if not isinstance(choices, list) or not choices:
        raise ValueError(f'OpenRouter response has no choices: {openrouter_response_summary(response)}')
    message = choices[0].get('message') if isinstance(choices[0], dict) else None
    content = message.get('content') if isinstance(message, dict) else None
    if not isinstance(content, str) or not content.strip():
        raise ValueError(f'OpenRouter returned empty text: {openrouter_response_summary(response)}')
    return content.strip()


def openrouter_text(*args, **kwargs) -> str:
    response = openrouter_chat(*args, **kwargs)
    return openrouter_message_content(response)


def json_candidate_text(text: str) -> str:
    stripped = text.strip()
    match = re.fullmatch(r'```(?:json)?\s*(.*?)\s*```', stripped, flags=re.DOTALL)
    if match:
        stripped = match.group(1).strip()
    start = stripped.find('{')
    end = stripped.rfind('}')
    if start != -1 and end != -1 and end > start:
        return stripped[start:end + 1]
    return stripped


def parse_json_text(text: str) -> dict[str, Any]:
    decoded = json.loads(json_candidate_text(text))
    if not isinstance(decoded, dict):
        raise ValueError('Expected a JSON object')
    return decoded


def try_local_json_repair(text: str) -> tuple[dict[str, Any], str] | None:
    try:
        from json_repair import repair_json
    except Exception:
        return None
    try:
        repaired_value = repair_json(json_candidate_text(text))
        if isinstance(repaired_value, str):
            repaired_text = repaired_value
            decoded = json.loads(repaired_text)
        else:
            decoded = repaired_value
            repaired_text = json.dumps(decoded, indent=2, ensure_ascii=False)
        if not isinstance(decoded, dict):
            return None
        return decoded, repaired_text
    except Exception:
        return None


def safe_artifact_name(label: str) -> str:
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', label.strip()).strip('_.-')
    return safe or 'llm_json'


def largest_token_cap(*caps: int | None) -> int | None:
    numeric_caps = [cap for cap in caps if cap is not None]
    return max(numeric_caps) if numeric_caps else None


def openrouter_json(
    model: str,
    messages: list[dict[str, Any]],
    *,
    label: str,
    max_tokens: int | None,
    temperature: float = 0,
) -> dict[str, Any]:
    response = openrouter_chat(
        model,
        messages,
        temperature=temperature,
        response_format={'type': 'json_object'},
        max_tokens=max_tokens,
    )
    text = openrouter_message_content(response)
    try:
        return parse_json_text(text)
    except (json.JSONDecodeError, ValueError) as exc:
        artifact_label = safe_artifact_name(label)
        raw_path = write_text(f'raw_{artifact_label}_invalid_json.txt', text)
        summary_path = write_json(
            f'raw_{artifact_label}_response_summary.json',
            openrouter_response_summary(response),
        )

        local_repair = try_local_json_repair(text)
        if local_repair is not None:
            repaired, repaired_text = local_repair
            write_text(f'raw_{artifact_label}_local_repaired_json.txt', repaired_text)
            return repaired

        repair_prompt = f'''
The text below was intended to be a JSON object but is invalid.
Repair it into valid JSON only. Preserve the intended fields and omit any item that cannot be safely repaired.

JSON parser error:
{exc}

Invalid JSON text:
{text}
'''.strip()
        try:
            repaired_text = openrouter_text(
                model,
                [
                    {'role': 'system', 'content': 'You repair malformed JSON. Return valid JSON only.'},
                    {'role': 'user', 'content': repair_prompt},
                ],
                temperature=0,
                response_format={'type': 'json_object'},
                max_tokens=largest_token_cap(max_tokens, JSON_REPAIR_MAX_TOKENS),
            )
        except Exception as llm_repair_exc:
            raise RuntimeError(
                f'Could not parse JSON for {label}. Raw response saved to {raw_path}. '
                f'Response summary saved to {summary_path}. Local json-repair was unavailable '
                f'or failed, and LLM repair failed: {llm_repair_exc}'
            ) from llm_repair_exc

        try:
            repaired = parse_json_text(repaired_text)
        except Exception as repair_exc:
            repaired_path = write_text(f'raw_{artifact_label}_repair_failed.txt', repaired_text)
            raise RuntimeError(
                f'Could not parse or repair JSON for {label}. '
                f'Raw response saved to {raw_path}; response summary saved to {summary_path}; '
                f'repair response saved to {repaired_path}.'
            ) from repair_exc
        write_text(f'raw_{artifact_label}_repaired_json.txt', repaired_text)
        return repaired


def format_bytes(size: int) -> str:
    units = ['B', 'KB', 'MB', 'GB']
    value = float(size)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.1f} {unit}' if unit != 'B' else f'{int(value)} B'
        value /= 1024
    return f'{size} B'


def verify_written_file(path: Path, label: str) -> Path:
    path = path.resolve()
    if not path.exists():
        raise RuntimeError(f'{label} was expected at {path}, but the file does not exist.')
    size = path.stat().st_size
    if size <= 0:
        raise RuntimeError(f'{label} was written to {path}, but the file is empty.')
    return path


def write_text(name: str, text: str) -> Path:
    path = ARTIFACT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    return verify_written_file(path, name)


def write_json(name: str, payload: Any) -> Path:
    path = ARTIFACT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    return verify_written_file(path, name)


def chunk_text(text: str, max_chars: int) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    paragraphs = re.split(r'\n{2,}', text)
    chunks: list[str] = []
    current: list[str] = []
    current_len = 0
    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if current and current_len + len(paragraph) + 2 > max_chars:
            chunks.append('\n\n'.join(current))
            current = []
            current_len = 0
        if len(paragraph) > max_chars:
            chunks.extend(textwrap.wrap(paragraph, width=max_chars, break_long_words=False, replace_whitespace=False))
            continue
        current.append(paragraph)
        current_len += len(paragraph) + 2
    if current:
        chunks.append('\n\n'.join(current))
    return chunks



def normalize_query_text(text: str) -> str:
    return " ".join(str(text).split())


def truncate_query_text(text: str, limit: int) -> str:
    normalized = normalize_query_text(text)
    if len(normalized) <= limit:
        return normalized
    truncated = normalized[: max(0, limit - 3)].rsplit(" ", 1)[0].rstrip()
    return f"{truncated}..." if truncated else normalized[:limit]


def prepare_ai_baseline_query(query: str) -> str:
    limit = min(AI_BASELINE_MAX_QUERY_CHARS, AI_BASELINE_QUERY_CHAR_BUDGET)
    prepared = truncate_query_text(query, limit)
    if len(prepared) > AI_BASELINE_MAX_QUERY_CHARS:
        raise ValueError(
            f"AI Baseline query is {len(prepared)} chars, above limit {AI_BASELINE_MAX_QUERY_CHARS}."
        )
    return prepared


def ai_baseline_headers() -> dict[str, str]:
    return {'X-API-Key': require_env(AI_BASELINE_API_KEY_ENV)}


def call_ai_baseline(query: str) -> dict[str, Any]:
    query = prepare_ai_baseline_query(query)
    if not AI_BASELINE_API_BASE_URL:
        raise RuntimeError('Set AI_BASELINE_API_BASE_URL in the first notebook cell.')
    payload: dict[str, Any] = {
        'query': query,
        'mode': AI_BASELINE_MODE,
        'effort': AI_BASELINE_EFFORT,
        'include': {
            'answer': False,
            'evidence': True,
            'evidence_instructions': True,
            'agent_context': True,
            'summary': True,
        },
    }
    max_evidence = globals().get('AI_BASELINE_MAX_EVIDENCE', 800)
    if max_evidence is not None:
        payload['retrieval'] = {'max_evidence': max_evidence}
    return post_json(
        f'{AI_BASELINE_API_BASE_URL}/{AI_BASELINE_DOMAIN}/query',
        ai_baseline_headers(),
        payload,
        AI_BASELINE_TIMEOUT_SECONDS,
    )


def extract_agent_context(response: dict[str, Any]) -> str:
    agent_context = response.get('agent_context')
    if not isinstance(agent_context, dict):
        raise ValueError('AI Baseline response did not include agent_context')
    text = agent_context.get('text')
    if not isinstance(text, str) or not text.strip():
        raise ValueError('AI Baseline agent_context.text is empty')
    return text.strip()

In [ ]:
source_progress = NotebookProgress('Load source text', 3, 'Resolving input path')
input_source_path = notebook_path(INPUT_TEXT_PATH)
if not input_source_path.exists():
    raise FileNotFoundError(
        f'Input file not found: {input_source_path}. '
        'Place the file next to the notebook or update INPUT_TEXT_PATH.'
    )

source_progress.update(1, 'Reading source file')
source_text = input_source_path.read_text(encoding='utf-8').strip()
if not source_text:
    raise ValueError(f'Input file is empty: {input_source_path}')
source_progress.update(2, 'Saving normalized source artifact')
normalized_source_path = write_text('source_text.md', source_text)
source_progress.finish(f'Loaded {len(source_text):,} characters')

preview = source_text[:SOURCE_PREVIEW_CHARS].replace('```', '` ` `')
display(Markdown(
    f'## Source Text\n\nLoaded `{input_source_path}` and saved a normalized copy to `{normalized_source_path}`. '
    f'Source length: `{len(source_text):,}` characters.\n\n```text\n{preview}\n```'
))

In [ ]:
def focus_areas_text() -> str:
    if not FOCUS_AREAS:
        return 'No predefined focus areas. Infer the most relevant themes from the source text.'
    return '\n'.join(f'- {area}' for area in FOCUS_AREAS)


TOPIC_JSON_STYLE_RULES = '''
- Return compact JSON only, with no markdown or commentary.
- Do not copy long source passages or echo candidate batches verbatim.
- Use at most 3 short source_claims per topic.
- Use at most 3 short risks_or_unknowns per topic.
- Write investment_angle, why_it_could_matter, public_market_access,
  ai_baseline_grounding_strategy, and ai_baseline_query as one concise sentence each.
- Keep companies to directly named firms plus the most relevant public proxies.
'''.strip()


EXTRACTION_SYSTEM = f'''
You are an institutional investment research analyst. Extract claims, facts,
observations, and hypotheses from arbitrary source text when they could plausibly
matter to public-market investors.

Prefer specific companies, tickers, industries, product cycles, demand/supply changes,
regulatory shifts, capex trends, margin drivers, customer behavior, and competitive
positioning. Ignore generic advice and motivational filler. Preserve distinct themes
even when some named companies are private, foreign, or outside AI Baseline's
{EVIDENCE_CORPUS_DESCRIPTION} corpus. Separate source claims, proxy ideas, and evidence gaps.
Do not collapse the analysis into only the theme that is easiest to verify in filings.
Return JSON only.
'''.strip()


def extract_topics_from_chunk(chunk: str, chunk_index: int, chunk_count: int) -> dict[str, Any]:
    prompt = f'''
Analyze {DOCUMENT_LABEL} chunk {chunk_index} of {chunk_count}.
Return at most 7 topics from this chunk. Keep each string concise and avoid long quotations.

Focus areas:
{focus_areas_text()}

JSON style rules:
{TOPIC_JSON_STYLE_RULES}

Return this JSON shape:
{{
  "topics": [
    {{
      "topic": "short title",
      "topic_category": "short category inferred from the {DOCUMENT_LABEL}",
      "investment_angle": "one concise sentence on why an investor might care",
      "companies": [{{"name": "company name if stated", "ticker": "ticker if known or null"}}],
      "source_claims": ["up to 3 short claims or observations from the {DOCUMENT_LABEL}"],
      "why_it_could_matter": "one concise sentence on possible financial or valuation implication",
      "risks_or_unknowns": ["up to 3 short points that need verification"],
      "public_market_access": "one concise sentence: direct public company, public proxy, or out-of-corpus/private/foreign",
      "ai_baseline_grounding_strategy": "one concise sentence: direct evidence, proxy evidence, or likely out-of-corpus; describe what to search for",
      "ai_baseline_query": "one concise focused AI Baseline search query against {EVIDENCE_CORPUS_DESCRIPTION}; use proxies if direct named companies are private, foreign, or out of corpus"
    }}
  ]
}}

Source chunk:
{chunk}
'''.strip()
    return openrouter_json(
        OPENROUTER_LLM_MODEL,
        [
            {'role': 'system', 'content': EXTRACTION_SYSTEM},
            {'role': 'user', 'content': prompt},
        ],
        label=f'extract_chunk_{chunk_index}',
        max_tokens=EXTRACTION_MAX_TOKENS,
    )


def topic_quality_warnings(topics_payload: dict[str, Any]) -> list[str]:
    topics = topics_payload.get('topics') if isinstance(topics_payload.get('topics'), list) else []
    warnings: list[str] = []
    min_topics = min(MAX_TOPICS, MIN_TOPICS)
    if len(topics) < min_topics:
        warnings.append(f'Only {len(topics)} topic(s); expected at least {min_topics}.')
    categories = {
        str(topic.get('topic_category') or topic.get('topic') or '').strip().lower()
        for topic in topics
        if isinstance(topic, dict)
    }
    if len(topics) >= min_topics and len(categories) < min_topics:
        warnings.append('Topics are not diverse enough; several appear to describe the same category.')
    return warnings


def repair_topic_quality(
    candidate_batches: list[dict[str, Any]],
    topics_payload: dict[str, Any],
    warnings: list[str],
) -> dict[str, Any]:
    prompt = f'''
The consolidated topic list failed the notebook's quality checks:
{json.dumps(warnings, indent=2, ensure_ascii=False)}

Rewrite the consolidated topics using the candidate batches below. Return valid JSON only.

Hard requirements:
- Return between {min(MAX_TOPICS, MIN_TOPICS)} and {MAX_TOPICS} topics.
- Preserve distinct source-supported themes instead of over-compressing them.
- If focus areas are provided, cover them when the source text supports them.
- Keep private, foreign, and out-of-corpus companies visible as source claims, with public proxy / AI Baseline grounding strategy when possible.
- Keep the same JSON shape and field names as the extraction step.
- Apply the JSON style rules below.

JSON style rules:
{TOPIC_JSON_STYLE_RULES}

Focus areas:
{focus_areas_text()}

Current consolidated topics:
{json.dumps(topics_payload, indent=2, ensure_ascii=False)}

Candidate batches:
{json.dumps(candidate_batches, ensure_ascii=False)}
'''.strip()
    return openrouter_json(
        OPENROUTER_LLM_MODEL,
        [
            {'role': 'system', 'content': EXTRACTION_SYSTEM},
            {'role': 'user', 'content': prompt},
        ],
        label='repair_topic_quality',
        max_tokens=CONSOLIDATION_MAX_TOKENS,
    )


def consolidate_topics(candidate_batches: list[dict[str, Any]]) -> dict[str, Any]:
    prompt = f'''
Deduplicate and rank the candidate investment topics below. Return between {min(MAX_TOPICS, MIN_TOPICS)} and {MAX_TOPICS} topics.

Focus areas:
{focus_areas_text()}

Hard rules:
- Do not collapse the output into the single theme most verifiable by AI Baseline.
- Include private, foreign, and out-of-corpus companies when central to the source, but mark them as source claims and provide a public-market proxy / AI Baseline grounding strategy.
- Favor specific, actionable, material themes, but do not discard a useful theme solely because AI Baseline can only retrieve proxy evidence.
- Return the same JSON shape with a top-level "topics" array. Preserve useful source claims and write strong AI Baseline queries.
- Apply the JSON style rules below.

JSON style rules:
{TOPIC_JSON_STYLE_RULES}

Candidate batches:
{json.dumps(candidate_batches, ensure_ascii=False)}
'''.strip()
    return openrouter_json(
        OPENROUTER_LLM_MODEL,
        [
            {'role': 'system', 'content': EXTRACTION_SYSTEM},
            {'role': 'user', 'content': prompt},
        ],
        label='consolidate_topics',
        max_tokens=CONSOLIDATION_MAX_TOKENS,
    )


chunks = chunk_text(source_text, EXTRACTION_CHUNK_CHARS)
extraction_progress = NotebookProgress('Extract investment topics', len(chunks) + 2, f'Prepared {len(chunks)} source chunk(s)')
candidate_batches = []
for index, chunk in enumerate(chunks, start=1):
    extraction_progress.update(index - 1, f'Extracting chunk {index}/{len(chunks)}')
    candidate_batches.append(extract_topics_from_chunk(chunk, index, len(chunks)))
    extraction_progress.update(index, f'Finished chunk {index}/{len(chunks)}')

extraction_progress.update(len(chunks), 'Consolidating and ranking topics')
topics_payload = consolidate_topics(candidate_batches)
quality_warnings = topic_quality_warnings(topics_payload)
if quality_warnings:
    extraction_progress.update(len(chunks) + 1, 'Repairing topic list')
    topics_payload = repair_topic_quality(candidate_batches, topics_payload, quality_warnings)
topics = topics_payload.get('topics') if isinstance(topics_payload.get('topics'), list) else []
extraction_progress.finish(f'Extracted {len(topics)} topic(s)')
topics_path = write_json('investment_topics.json', topics_payload)

topics_preview = json.dumps(topics_payload, indent=2, ensure_ascii=False)[:TOPICS_PREVIEW_CHARS].replace('```', '` ` `')
display(Markdown(
    f'## Extracted Investment Topics\n\nSaved to `{topics_path}`. Topic count: `{len(topics)}`.\n\n'
    f'```json\n{topics_preview}\n```'
))

In [ ]:

def topic_company_text(topic: dict[str, Any]) -> str:
    companies = topic.get('companies')
    if not isinstance(companies, list):
        return ''
    labels = []
    for company in companies:
        if not isinstance(company, dict):
            continue
        name = str(company.get('name') or '').strip()
        ticker = str(company.get('ticker') or '').strip()
        if name and ticker and ticker.lower() != 'null':
            labels.append(f'{name} ({ticker})')
        elif name:
            labels.append(name)
        elif ticker and ticker.lower() != 'null':
            labels.append(ticker)
    return ', '.join(labels)


def build_ai_baseline_search_query(topic: dict[str, Any], index: int) -> str:
    supplied_query = topic.get('ai_baseline_query')
    if isinstance(supplied_query, str) and supplied_query.strip():
        base = supplied_query.strip()
    else:
        base = ' '.join(
            str(topic.get(field) or '').strip()
            for field in ('topic', 'investment_angle', 'why_it_could_matter')
            if str(topic.get(field) or '').strip()
        )
    companies = topic_company_text(topic)
    company_clause = f' Named companies from source text: {companies}.' if companies else ''
    access = str(topic.get('public_market_access') or '').strip()
    strategy = str(topic.get('ai_baseline_grounding_strategy') or '').strip()
    strategy_clause = f' Evidence strategy: {strategy}.' if strategy else ''
    access_clause = f' Public-market access: {access}.' if access else ''
    query = (
        f'Use AI Baseline as a search engine over {EVIDENCE_CORPUS_DESCRIPTION}. Find direct evidence or proxy evidence relevant to this source-derived investment theme: '
        f'{base}.{company_clause}{access_clause}{strategy_clause} '
        'If named companies are private, foreign, or outside the corpus, search for suppliers, customers, competitors, substitutes, and risk disclosures instead of assuming direct coverage. '
        'Prioritize source evidence on revenue, margins, capex, supply chain, competition, demand, risk factors, and catalysts.'
    )
    return prepare_ai_baseline_query(query)


def build_ai_baseline_search_queries(topics_payload: dict[str, Any]) -> list[dict[str, str]]:
    topics = topics_payload.get('topics') if isinstance(topics_payload.get('topics'), list) else []
    searches = []
    for index, topic in enumerate(topics[:MAX_AI_BASELINE_TOPIC_QUERIES], start=1):
        if not isinstance(topic, dict):
            continue
        title = str(topic.get('topic') or f'Topic {index}').strip()
        searches.append({
            'topic': title,
            'query': build_ai_baseline_search_query(topic, index),
        })
    if not searches:
        searches.append({
            'topic': 'Source investment themes',
            'query': prepare_ai_baseline_query(
                f'Find evidence in {EVIDENCE_CORPUS_DESCRIPTION} relevant to the main investment themes in the source text.'
            ),
        })
    return searches


def plan_ai_baseline_followups(
    topics_payload: dict[str, Any],
    initial_queries: list[dict[str, str]],
    initial_context: str,
) -> dict[str, Any]:
    prompt = f'''
Review the initial AI Baseline retrieval contexts and decide whether targeted follow-up AI Baseline search queries are needed.

Return JSON only with this shape:
{{
  "coverage_summary": "what the current AI Baseline context covers well",
  "topic_coverage": [
    {{"topic": "source topic title", "coverage": "direct | proxy | gap", "notes": "how AI Baseline covers or misses it"}}
  ],
  "evidence_gaps": ["material gap, contradiction, or unresolved investment question"],
  "followup_queries": [
    {{
      "query": "concise standalone AI Baseline search query under {AI_BASELINE_QUERY_CHAR_BUDGET} characters",
      "rationale": "why this query should improve the final investment note"
    }}
  ]
}}

Rules:
- Return at most {MAX_AI_BASELINE_FOLLOWUP_QUERIES} follow-up queries.
- Treat AI Baseline as a search/retrieval system. Queries must be concise standalone search questions, not long prompts.
- Do not ask broad duplicate queries if the initial context already covers the issue.
- Prefer specific company, sector, risk, margin, capex, demand, regulatory, or catalyst checks.
- Explicitly label each source-derived topic as directly covered, proxy-covered, or still a gap.
- If a topic is out of corpus, generate proxy follow-ups around S&P 500 suppliers, customers, competitors, or risk disclosures rather than repeating unavailable private-company searches.
- Return an empty followup_queries list when the initial context is sufficient.

Source-derived topics:
{json.dumps(topics_payload, indent=2, ensure_ascii=False)}

Initial AI Baseline search queries:
{json.dumps(initial_queries, indent=2, ensure_ascii=False)}

Initial AI Baseline context excerpt:
{initial_context[:AI_BASELINE_AUDIT_CONTEXT_CHARS]}
'''.strip()
    return openrouter_json(
        OPENROUTER_LLM_MODEL,
        [
            {'role': 'system', 'content': 'You are a rigorous investment research editor. Return JSON only.'},
            {'role': 'user', 'content': prompt},
        ],
        label='ai_baseline_followup_plan',
        max_tokens=AI_BASELINE_AUDIT_MAX_TOKENS,
    )


def planned_followup_queries(plan: dict[str, Any]) -> list[dict[str, str]]:
    raw_queries = plan.get('followup_queries')
    if not isinstance(raw_queries, list):
        return []
    queries = []
    for item in raw_queries[:MAX_AI_BASELINE_FOLLOWUP_QUERIES]:
        if isinstance(item, dict):
            query = item.get('query')
            rationale = item.get('rationale')
        else:
            query = str(item)
            rationale = ''
        if isinstance(query, str) and query.strip():
            queries.append({
                'query': prepare_ai_baseline_query(query),
                'rationale': str(rationale or '').strip(),
            })
    return queries


def combined_ai_baseline_context(initial_results: list[dict[str, Any]], followups: list[dict[str, Any]]) -> str:
    sections = []
    for index, result in enumerate(initial_results, start=1):
        sections.extend([
            f'# AI Baseline search context {index}: {result["topic"]}',
            f'Query: {result["query"]}',
            result['agent_context'],
        ])
    for index, followup in enumerate(followups, start=1):
        sections.extend([
            f'# Follow-up AI Baseline search context {index}',
            f"Query: {followup['query']}",
            f"Rationale: {followup.get('rationale', '')}",
            followup['agent_context'],
        ])
    return '\n\n'.join(sections)


ai_baseline_search_queries = build_ai_baseline_search_queries(topics_payload)
ai_baseline_queries_path = write_json('ai_baseline_search_queries.json', ai_baseline_search_queries)

ai_baseline_progress_total = len(ai_baseline_search_queries) + 2
ai_baseline_progress = NotebookProgress(
    'AI Baseline retrieval loop',
    ai_baseline_progress_total,
    f'Prepared {len(ai_baseline_search_queries)} search query(s)',
)

initial_ai_baseline_results: list[dict[str, Any]] = []
for index, search in enumerate(ai_baseline_search_queries, start=1):
    ai_baseline_progress.update(index, f'Running search query {index}/{len(ai_baseline_search_queries)}')
    response = call_ai_baseline(search['query'])
    context = extract_agent_context(response)
    summary = response.get('summary') if isinstance(response.get('summary'), dict) else {}
    initial_ai_baseline_results.append({
        'topic': search['topic'],
        'query': search['query'],
        'request_id': response.get('request_id'),
        'evidence_count': summary.get('evidence_count'),
        'agent_context_chars': len(context),
        'agent_context': context,
        'response': response,
    })
    ai_baseline_progress.update(index, f'Finished search query {index}/{len(ai_baseline_search_queries)}')

initial_context = combined_ai_baseline_context(initial_ai_baseline_results, [])
initial_results_path = write_json('ai_baseline_initial_results.json', initial_ai_baseline_results)
initial_context_path = write_text('ai_baseline_initial_agent_context.md', initial_context)

ai_baseline_followup_plan: dict[str, Any] = {'followup_queries': []}
ai_baseline_followups: list[dict[str, Any]] = []
if ENABLE_AI_BASELINE_FOLLOWUPS:
    ai_baseline_progress.update(len(ai_baseline_search_queries) + 1, 'Auditing retrieval results for evidence gaps')
    ai_baseline_followup_plan = plan_ai_baseline_followups(
        topics_payload,
        ai_baseline_search_queries,
        initial_context,
    )
    followup_queries = planned_followup_queries(ai_baseline_followup_plan)
    ai_baseline_progress_total = len(ai_baseline_search_queries) + 2 + len(followup_queries)
    ai_baseline_progress.set_total(
        ai_baseline_progress_total,
        f'Planned {len(followup_queries)} follow-up search query(s)',
    )
    for index, followup in enumerate(followup_queries, start=1):
        ai_baseline_progress.update(
            len(ai_baseline_search_queries) + 1 + index,
            f'Running follow-up search {index}/{len(followup_queries)}',
        )
        followup_response = call_ai_baseline(followup['query'])
        followup_context = extract_agent_context(followup_response)
        followup_summary = followup_response.get('summary') if isinstance(followup_response.get('summary'), dict) else {}
        ai_baseline_followups.append({
            'query': followup['query'],
            'rationale': followup.get('rationale', ''),
            'request_id': followup_response.get('request_id'),
            'evidence_count': followup_summary.get('evidence_count'),
            'agent_context_chars': len(followup_context),
            'agent_context': followup_context,
            'response': followup_response,
        })
else:
    ai_baseline_progress.update(len(ai_baseline_search_queries) + 1, 'Skipping follow-up audit')

ai_baseline_progress.update(ai_baseline_progress_total, 'Combining retrieval contexts and writing artifacts')
agent_context = combined_ai_baseline_context(initial_ai_baseline_results, ai_baseline_followups)
followup_plan_path = write_json('ai_baseline_followup_plan.json', ai_baseline_followup_plan)
followups_path = write_json('ai_baseline_followups.json', ai_baseline_followups)
combined_context_path = write_text('ai_baseline_combined_agent_context.md', agent_context)
ai_baseline_progress.finish(f'Completed {len(initial_ai_baseline_results)} search query(s) and {len(ai_baseline_followups)} follow-up query(s)')

total_evidence = sum(int(result.get('evidence_count') or 0) for result in [*initial_ai_baseline_results, *ai_baseline_followups])
context_preview = agent_context[:AI_BASELINE_PREVIEW_CHARS].replace('```', '` ` `')
display(Markdown(
    f'## AI Baseline Retrieval\n\nSaved search queries to `{ai_baseline_queries_path}`, initial results to `{initial_results_path}`, '
    f'initial context to `{initial_context_path}`, follow-up plan to `{followup_plan_path}`, '
    f'follow-up responses to `{followups_path}`, and combined context to `{combined_context_path}`.\n\n'
    f'Initial search queries run: `{len(initial_ai_baseline_results)}`  \n'
    f'Follow-up queries run: `{len(ai_baseline_followups)}`  \n'
    f'Total reported evidence count: `{total_evidence}`  \n'
    f'Combined context length: `{len(agent_context):,}` characters.\n\n'
    f'```text\n{context_preview}\n```'
))

In [ ]:
ANSWER_SYSTEM = f'''
You are an institutional investment research analyst. Write an investment-style research
note in clear, decision-useful language, but do not present it as personalized financial advice. Ground it
in the source-derived topics and combined AI Baseline context. Distinguish clearly between
source claims, AI Baseline-supported facts, proxy evidence, and hypotheses. Do not invent
citations or data. Do not let lack of evidence in {EVIDENCE_CORPUS_DESCRIPTION} erase an
important source theme; label it as proxy-only or unverified instead.
'''.strip()


def truncate_preserving_lines(text: str, max_chars: int) -> str:
    if max_chars <= 0 or len(text) <= max_chars:
        return text
    truncated = text[:max_chars].rsplit('\n', 1)[0].rstrip()
    return truncated + f'\n\n[Truncated to {max_chars:,} characters for prompt budget.]'


def split_ai_baseline_context_sections(context: str) -> list[str]:
    sections = re.split(
        r'(?=^# (?:AI Baseline search context|Follow-up AI Baseline search context) )',
        context,
        flags=re.MULTILINE,
    )
    return [section.strip() for section in sections if section.strip()]


def compact_source_registry(section: str, max_chars: int) -> str:
    match = re.search(r'<source_registry>.*?</source_registry>', section, flags=re.DOTALL)
    if not match or max_chars <= 0:
        return ''
    source_lines = re.findall(r'^\s*<source\b.*?/>', match.group(0), flags=re.MULTILINE)
    if not source_lines:
        return truncate_preserving_lines(match.group(0), max_chars)
    compact = '<source_registry>\n' + '\n'.join(source_lines) + '\n</source_registry>'
    return truncate_preserving_lines(compact, max_chars)


def evidence_entries(section: str) -> list[str]:
    entries = re.findall(r'<entry\b.*?</entry>', section, flags=re.DOTALL)
    if entries:
        return entries
    chains = re.findall(r'<evidence_chain\b.*?</evidence_chain>', section, flags=re.DOTALL)
    return chains


def compact_evidence_entries(section: str, max_entries: int, max_chars: int) -> str:
    selected = []
    used = 0
    for entry in evidence_entries(section):
        cleaned = re.sub(r'\n{3,}', '\n\n', entry.strip())
        if len(selected) >= max_entries:
            break
        if selected and used + len(cleaned) + 2 > max_chars:
            break
        if not selected and len(cleaned) > max_chars:
            selected.append(truncate_preserving_lines(cleaned, max_chars))
            break
        selected.append(cleaned)
        used += len(cleaned) + 2
    if not selected:
        evidence_start = section.find('## Evidence')
        fallback = section[evidence_start:] if evidence_start != -1 else section
        return truncate_preserving_lines(fallback, max_chars)
    omitted = max(0, len(evidence_entries(section)) - len(selected))
    suffix = f'\n\n[Omitted {omitted:,} lower-ranked evidence entries for prompt budget.]' if omitted else ''
    return '<selected_evidence_entries>\n' + '\n\n'.join(selected) + suffix + '\n</selected_evidence_entries>'


def compact_ai_baseline_section(section: str, per_section_chars: int) -> str:
    lines = section.splitlines()
    title = lines[0] if lines else '# AI Baseline context'
    query_lines = []
    for line in lines[1:]:
        if line.startswith('## Evidence Instructions') or line.startswith('## Evidence'):
            break
        query_lines.append(line)
    query_text = '\n'.join(query_lines).strip()

    source_budget = min(ANSWER_CONTEXT_SOURCE_REGISTRY_MAX_CHARS, max(0, per_section_chars // 4))
    evidence_budget = max(4_000, per_section_chars - len(title) - len(query_text) - source_budget - 500)
    parts = [
        title,
        query_text,
        'Use source IDs in brackets exactly as shown below when citing retrieved filing evidence.',
    ]
    source_registry = compact_source_registry(section, source_budget)
    if source_registry:
        parts.append(source_registry)
    parts.append(compact_evidence_entries(
        section,
        ANSWER_CONTEXT_MAX_EVIDENCE_ENTRIES_PER_SEARCH,
        evidence_budget,
    ))
    return truncate_preserving_lines('\n\n'.join(part for part in parts if part), per_section_chars)


def compact_ai_baseline_context(context: str) -> str:
    sections = split_ai_baseline_context_sections(context)
    if not sections:
        return truncate_preserving_lines(context, ANSWER_CONTEXT_MAX_CHARS)
    per_section_chars = min(
        ANSWER_CONTEXT_PER_SEARCH_MAX_CHARS,
        max(8_000, ANSWER_CONTEXT_MAX_CHARS // max(1, len(sections))),
    )
    compacted_sections = [compact_ai_baseline_section(section, per_section_chars) for section in sections]
    compacted = '\n\n'.join(compacted_sections)
    return truncate_preserving_lines(compacted, ANSWER_CONTEXT_MAX_CHARS)


answer_context = compact_ai_baseline_context(agent_context)
answer_context_path = write_text('answer_context.md', answer_context)
answer_context_report = {
    'full_context_chars': len(agent_context),
    'answer_context_chars': len(answer_context),
    'answer_context_max_chars': ANSWER_CONTEXT_MAX_CHARS,
    'per_search_max_chars': ANSWER_CONTEXT_PER_SEARCH_MAX_CHARS,
    'max_evidence_entries_per_search': ANSWER_CONTEXT_MAX_EVIDENCE_ENTRIES_PER_SEARCH,
    'source_registry_max_chars': ANSWER_CONTEXT_SOURCE_REGISTRY_MAX_CHARS,
    'context_sections': len(split_ai_baseline_context_sections(agent_context)),
    'answer_context_path': str(answer_context_path),
}
answer_context_report_path = write_json('answer_context_report.json', answer_context_report)


final_prompt = f'''
Create an investment-style research note from the material below.

Input label:
{DOCUMENT_LABEL}

Required structure: use these exact H2 markdown headings, in this order:
{chr(10).join(f'## {heading}' for heading in REQUIRED_ANSWER_SECTION_HEADINGS)}

Use concise, investor-facing language. Include source references only when present in the
combined AI Baseline context. If evidence is weak, say so directly.
Before ranking insights, map each extracted source topic to one of: directly supported by
AI Baseline, proxy-supported, or not supported/out-of-corpus. Keep important unverified
source themes visible as hypotheses rather than turning the whole note into the single
best-supported angle.

Source-derived investment topics:
{json.dumps(topics_payload, indent=2, ensure_ascii=False)}

AI Baseline follow-up audit:
{json.dumps(ai_baseline_followup_plan, indent=2, ensure_ascii=False)}

Compacted AI Baseline agent context:
{answer_context}

Full raw AI Baseline context is saved separately at `{combined_context_path}`. The compacted context used for this answer is saved at `{answer_context_path}` and summarized at `{answer_context_report_path}`.
'''.strip()

def missing_required_answer_sections(note: str) -> list[str]:
    missing = []
    for heading in REQUIRED_ANSWER_SECTION_HEADINGS:
        pattern = rf'^##\s+{re.escape(heading)}\s*$'
        if not re.search(pattern, note, flags=re.IGNORECASE | re.MULTILINE):
            missing.append(heading)
    return missing


def complete_missing_answer_sections(note: str, missing_sections: list[str]) -> str:
    completion_prompt = f'''
The investment-style research note below is missing required sections or stopped before the end.
Write only the missing sections listed below, using the exact H2 markdown headings provided.
Do not repeat sections that are already present. Do not invent citations or facts.

Input label:
{DOCUMENT_LABEL}

Missing sections:
{chr(10).join(f'## {section}' for section in missing_sections)}

Source-derived investment topics:
{json.dumps(topics_payload, indent=2, ensure_ascii=False)}

AI Baseline follow-up audit:
{json.dumps(ai_baseline_followup_plan, indent=2, ensure_ascii=False)}

Compacted AI Baseline context:
{answer_context}

Current note:
{note}
'''.strip()
    completion = openrouter_text(
        OPENROUTER_LLM_MODEL,
        [
            {'role': 'system', 'content': ANSWER_SYSTEM},
            {'role': 'user', 'content': completion_prompt},
        ],
        temperature=ANSWER_TEMPERATURE,
        max_tokens=ANSWER_COMPLETION_MAX_TOKENS,
    )
    return note.rstrip() + '\n\n' + completion.strip()


answer_progress = NotebookProgress('Generate investment note', 4, 'Preparing final prompt')
answer_progress.update(1, 'Calling OpenRouter LLM')
investment_note = openrouter_text(
    OPENROUTER_LLM_MODEL,
    [
        {'role': 'system', 'content': ANSWER_SYSTEM},
        {'role': 'user', 'content': final_prompt},
    ],
    temperature=ANSWER_TEMPERATURE,
    max_tokens=ANSWER_MAX_TOKENS,
)

missing_sections = missing_required_answer_sections(investment_note)
if missing_sections:
    answer_progress.update(2, 'Completing missing required sections')
    investment_note = complete_missing_answer_sections(investment_note, missing_sections)
    missing_sections = missing_required_answer_sections(investment_note)
if missing_sections:
    raise RuntimeError('Investment note is still missing required section(s): ' + ', '.join(missing_sections))

answer_progress.update(3, 'Writing investment note artifact')
investment_note_path = write_text('investment_note.md', investment_note)
investment_note_size = investment_note_path.stat().st_size
answer_progress.finish(f'Investment note written ({format_bytes(investment_note_size)})')
display(Markdown(
    f'## Investment Note\n\nSaved to `{investment_note_path}` '
    f'(`{format_bytes(investment_note_size)}`).\n\n' + investment_note
))

In [ ]:
def markdown_to_basic_html(markdown_text: str) -> str:
    try:
        import markdown as markdown_module

        return markdown_module.markdown(
            markdown_text,
            extensions=PDF_MARKDOWN_EXTENSIONS,
            output_format='html5',
        )
    except Exception as exc:
        if REQUIRE_RICH_PDF_MARKDOWN:
            raise RuntimeError(
                'Rich Markdown PDF rendering requires the `markdown` package. '
                'Install it with: uv pip install --python .venv/bin/python markdown'
            ) from exc

    lines = []
    in_list = False
    for raw_line in markdown_text.splitlines():
        line = raw_line.rstrip()
        stripped = line.strip()
        if not stripped:
            if in_list:
                lines.append('</ul>')
                in_list = False
            continue
        if stripped.startswith('#'):
            if in_list:
                lines.append('</ul>')
                in_list = False
            level = len(stripped) - len(stripped.lstrip('#'))
            level = min(max(level, 1), 4)
            text = stripped[level:].strip()
            lines.append(f'<h{level}>{html.escape(text)}</h{level}>')
        elif stripped.startswith('- '):
            if not in_list:
                lines.append('<ul>')
                in_list = True
            lines.append(f'<li>{html.escape(stripped[2:].strip())}</li>')
        else:
            if in_list:
                lines.append('</ul>')
                in_list = False
            lines.append(f'<p>{html.escape(stripped)}</p>')
    if in_list:
        lines.append('</ul>')
    return '\n'.join(lines)


def verify_pdf_file(path: Path, renderer: str) -> Path:
    path = verify_written_file(path, f'{renderer} PDF export')
    if path.read_bytes()[:5] != b'%PDF-':
        raise RuntimeError(f'{renderer} wrote {path}, but it does not look like a PDF file.')
    return path


def pdf_renderer_environment() -> dict[str, Any]:
    import shutil

    return {
        'weasyprint_importable': importlib.util.find_spec('weasyprint') is not None,
        'markdown_importable': importlib.util.find_spec('markdown') is not None,
        'reportlab_importable': importlib.util.find_spec('reportlab') is not None,
        'pandoc_path': shutil.which('pandoc'),
    }


def available_rich_pdf_renderers() -> list[str]:
    environment = pdf_renderer_environment()
    renderers = []
    if environment['weasyprint_importable']:
        renderers.append('weasyprint')
    if environment['reportlab_importable']:
        renderers.append('reportlab')
    if environment['pandoc_path']:
        renderers.append('pandoc')
    return renderers


def export_pdf_with_weasyprint(markdown_path: Path, pdf_path: Path) -> str:
    from weasyprint import HTML as WeasyHTML

    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    markdown_path = verify_written_file(markdown_path, 'investment_note.md input')
    markdown_text = markdown_path.read_text(encoding='utf-8')
    body = markdown_to_basic_html(markdown_text)
    html_document = f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <style>
    @page {{ size: Letter; margin: 0.75in; }}
    body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; font-size: 10.5pt; line-height: 1.45; color: #111827; }}
    h1 {{ font-size: 22pt; margin: 0 0 14pt; }}
    h2 {{ font-size: 15pt; margin: 18pt 0 8pt; border-bottom: 1px solid #d0d7de; padding-bottom: 3pt; }}
    h3 {{ font-size: 12pt; margin: 14pt 0 6pt; }}
    p, ul, ol {{ margin: 0 0 8pt; }}
    li {{ margin-bottom: 3pt; }}
    table {{ border-collapse: collapse; width: 100%; margin: 8pt 0 12pt; font-size: 8.5pt; table-layout: fixed; }}
    th, td {{ border: 1px solid #d0d7de; padding: 4pt 5pt; vertical-align: top; overflow-wrap: anywhere; }}
    th {{ background: #f6f8fa; font-weight: 700; }}
    code {{ font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 9pt; }}
    blockquote {{ border-left: 3pt solid #d0d7de; margin: 8pt 0; padding-left: 9pt; color: #4b5563; }}
  </style>
</head>
<body>
{body}
</body>
</html>
""".strip()
    WeasyHTML(string=html_document, base_url=str(markdown_path.parent)).write_pdf(str(pdf_path))
    return 'weasyprint'


def export_pdf_with_pandoc(markdown_path: Path, pdf_path: Path) -> str:
    import shutil
    import subprocess

    pandoc = shutil.which('pandoc')
    if not pandoc:
        raise RuntimeError('pandoc is not installed')
    result = subprocess.run(
        [pandoc, str(markdown_path), '-o', str(pdf_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        detail = (result.stderr or result.stdout or 'pandoc failed').strip()
        raise RuntimeError(detail[:1_000])
    return 'pandoc'


def pdf_safe_text(text: str) -> str:
    replacements = {
        '\u2013': '-',
        '\u2014': '-',
        '\u2018': "'",
        '\u2019': "'",
        '\u201c': '"',
        '\u201d': '"',
        '\u2022': '-',
        '\u00a0': ' ',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text.encode('latin-1', errors='replace').decode('latin-1')


def pdf_escape(text: str) -> str:
    text = pdf_safe_text(text)
    return text.replace('\\', '\\\\').replace('(', '\\(').replace(')', '\\)')


INLINE_MARKDOWN_PATTERN = re.compile(r'(`[^`\n]+`|\*\*[^*\n]+?\*\*)')


def reportlab_inline_markup(text: str) -> str:
    text = pdf_safe_text(text)
    parts = []
    last = 0
    for match in INLINE_MARKDOWN_PATTERN.finditer(text):
        parts.append(html.escape(text[last:match.start()], quote=False))
        token = match.group(0)
        if token.startswith('**') and token.endswith('**'):
            parts.append(f'<b>{html.escape(token[2:-2], quote=False)}</b>')
        elif token.startswith('`') and token.endswith('`'):
            parts.append(f'<font name="Courier">{html.escape(token[1:-1], quote=False)}</font>')
        last = match.end()
    parts.append(html.escape(text[last:], quote=False))
    return ''.join(parts)


def is_markdown_rule(line: str) -> bool:
    stripped = line.strip()
    return bool(re.fullmatch(r'[-*_]\s*[-*_\s]{2,}', stripped))


def markdown_table_cells(line: str) -> list[str]:
    stripped = line.strip()
    if not stripped.startswith('|'):
        return []
    return [cell.strip() for cell in stripped.strip('|').split('|')]


def is_markdown_table_separator(line: str) -> bool:
    cells = markdown_table_cells(line)
    return len(cells) >= 2 and all(re.fullmatch(r':?-{3,}:?', cell or '') for cell in cells)


def is_markdown_table_start(lines: list[str], index: int) -> bool:
    if index + 1 >= len(lines):
        return False
    return bool(markdown_table_cells(lines[index])) and is_markdown_table_separator(lines[index + 1])


def reportlab_markdown_flowables(markdown_text: str, styles: dict[str, Any], available_width: float) -> list[Any]:
    from reportlab.lib import colors
    from reportlab.platypus import HRFlowable, Paragraph, Preformatted, Spacer, Table, TableStyle

    lines = markdown_text.splitlines()
    flowables: list[Any] = []
    paragraph_lines: list[str] = []
    index = 0

    def flush_paragraph() -> None:
        if not paragraph_lines:
            return
        text = ' '.join(line.strip() for line in paragraph_lines if line.strip())
        if text:
            flowables.append(Paragraph(reportlab_inline_markup(text), styles['body']))
        paragraph_lines.clear()

    while index < len(lines):
        raw_line = lines[index]
        stripped = raw_line.strip()
        if not stripped:
            flush_paragraph()
            index += 1
            continue

        if stripped.startswith('```'):
            flush_paragraph()
            index += 1
            code_lines = []
            while index < len(lines) and not lines[index].strip().startswith('```'):
                code_lines.append(pdf_safe_text(lines[index]))
                index += 1
            if index < len(lines):
                index += 1
            flowables.append(Preformatted('\n'.join(code_lines), styles['code']))
            flowables.append(Spacer(1, 6))
            continue

        if is_markdown_table_start(lines, index):
            flush_paragraph()
            header = markdown_table_cells(lines[index])
            rows = [header]
            index += 2
            while index < len(lines) and markdown_table_cells(lines[index]):
                rows.append(markdown_table_cells(lines[index]))
                index += 1
            column_count = max(len(row) for row in rows)
            padded_rows = [row + [''] * (column_count - len(row)) for row in rows]
            cell_style = styles['table_cell']
            header_style = styles['table_header']
            table_data = [
                [
                    Paragraph(reportlab_inline_markup(cell), header_style if row_index == 0 else cell_style)
                    for cell in row
                ]
                for row_index, row in enumerate(padded_rows)
            ]
            table = Table(
                table_data,
                colWidths=[available_width / column_count] * column_count,
                repeatRows=1,
            )
            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#f3f4f6')),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.HexColor('#111827')),
                ('GRID', (0, 0), (-1, -1), 0.4, colors.HexColor('#d1d5db')),
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('LEFTPADDING', (0, 0), (-1, -1), 4),
                ('RIGHTPADDING', (0, 0), (-1, -1), 4),
                ('TOPPADDING', (0, 0), (-1, -1), 3),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
            ]))
            flowables.append(table)
            flowables.append(Spacer(1, 8))
            continue

        if is_markdown_rule(stripped):
            flush_paragraph()
            flowables.append(HRFlowable(width='100%', thickness=0.6, color=colors.HexColor('#d1d5db')))
            flowables.append(Spacer(1, 8))
            index += 1
            continue

        heading_match = re.match(r'^(#{1,4})\s+(.+)$', stripped)
        if heading_match:
            flush_paragraph()
            level = len(heading_match.group(1))
            style_name = {1: 'h1', 2: 'h2', 3: 'h3'}.get(level, 'h4')
            flowables.append(Paragraph(reportlab_inline_markup(heading_match.group(2)), styles[style_name]))
            index += 1
            continue

        bullet_match = re.match(r'^[-*]\s+(.+)$', stripped)
        if bullet_match:
            flush_paragraph()
            flowables.append(Paragraph(reportlab_inline_markup(bullet_match.group(1)), styles['body'], bulletText='-'))
            index += 1
            continue

        numbered_match = re.match(r'^(\d+)\.\s+(.+)$', stripped)
        if numbered_match:
            flush_paragraph()
            flowables.append(
                Paragraph(
                    reportlab_inline_markup(numbered_match.group(2)),
                    styles['body'],
                    bulletText=f'{numbered_match.group(1)}.',
                )
            )
            index += 1
            continue

        quote_match = re.match(r'^>\s*(.+)$', stripped)
        if quote_match:
            flush_paragraph()
            flowables.append(Paragraph(reportlab_inline_markup(quote_match.group(1)), styles['quote']))
            index += 1
            continue

        paragraph_lines.append(stripped)
        index += 1

    flush_paragraph()
    return flowables


def export_pdf_with_reportlab(markdown_path: Path, pdf_path: Path) -> str:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate

    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    markdown_path = verify_written_file(markdown_path, 'investment_note.md input')
    markdown_text = markdown_path.read_text(encoding='utf-8')
    margin = 0.68 * inch
    doc = SimpleDocTemplate(
        str(pdf_path),
        pagesize=letter,
        leftMargin=margin,
        rightMargin=margin,
        topMargin=margin,
        bottomMargin=margin,
        title=markdown_path.stem,
    )
    base = getSampleStyleSheet()
    styles = {
        'body': ParagraphStyle(
            'AIBaselineBody',
            parent=base['BodyText'],
            fontName='Helvetica',
            fontSize=9.6,
            leading=12.8,
            spaceAfter=6,
        ),
        'h1': ParagraphStyle(
            'AIBaselineH1',
            parent=base['Heading1'],
            fontName='Helvetica-Bold',
            fontSize=19,
            leading=23,
            spaceBefore=4,
            spaceAfter=12,
            textColor=colors.HexColor('#111827'),
        ),
        'h2': ParagraphStyle(
            'AIBaselineH2',
            parent=base['Heading2'],
            fontName='Helvetica-Bold',
            fontSize=14,
            leading=18,
            spaceBefore=12,
            spaceAfter=8,
            textColor=colors.HexColor('#111827'),
        ),
        'h3': ParagraphStyle(
            'AIBaselineH3',
            parent=base['Heading3'],
            fontName='Helvetica-Bold',
            fontSize=11.5,
            leading=15,
            spaceBefore=8,
            spaceAfter=5,
            textColor=colors.HexColor('#111827'),
        ),
        'h4': ParagraphStyle(
            'AIBaselineH4',
            parent=base['Heading4'],
            fontName='Helvetica-Bold',
            fontSize=10.5,
            leading=14,
            spaceBefore=6,
            spaceAfter=4,
            textColor=colors.HexColor('#111827'),
        ),
        'quote': ParagraphStyle(
            'AIBaselineQuote',
            parent=base['BodyText'],
            fontName='Helvetica',
            fontSize=9.2,
            leading=12.2,
            leftIndent=12,
            borderPadding=5,
            borderWidth=0.6,
            borderColor=colors.HexColor('#d1d5db'),
            textColor=colors.HexColor('#4b5563'),
            spaceAfter=7,
        ),
        'code': ParagraphStyle(
            'AIBaselineCode',
            parent=base['Code'],
            fontName='Courier',
            fontSize=8,
            leading=10,
            backColor=colors.HexColor('#f3f4f6'),
            borderPadding=5,
            spaceAfter=7,
        ),
        'table_cell': ParagraphStyle(
            'AIBaselineTableCell',
            parent=base['BodyText'],
            fontName='Helvetica',
            fontSize=7.3,
            leading=9.2,
        ),
        'table_header': ParagraphStyle(
            'AIBaselineTableHeader',
            parent=base['BodyText'],
            fontName='Helvetica-Bold',
            fontSize=7.4,
            leading=9.4,
        ),
    }
    flowables = reportlab_markdown_flowables(markdown_text, styles, doc.width)
    doc.build(flowables)
    return 'reportlab'


def plain_pdf_inline_text(text: str) -> str:
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'`([^`]+)`', r'\1', text)
    return text


def markdown_lines_for_plain_pdf(markdown_text: str) -> list[tuple[str, str, int]]:
    rendered: list[tuple[str, str, int]] = []
    for raw_line in markdown_text.splitlines():
        stripped = raw_line.strip()
        if not stripped:
            rendered.append(('', 'regular', 10))
            continue
        if is_markdown_table_separator(stripped):
            continue
        table_cells = markdown_table_cells(stripped)
        if table_cells:
            rendered.append((' | '.join(plain_pdf_inline_text(cell) for cell in table_cells), 'regular', 9))
            continue
        if is_markdown_rule(stripped):
            rendered.append(('_' * 72, 'regular', 10))
            continue
        if stripped.startswith('# '):
            rendered.append((plain_pdf_inline_text(stripped[2:].strip()), 'bold', 18))
        elif stripped.startswith('## '):
            rendered.append((plain_pdf_inline_text(stripped[3:].strip()), 'bold', 14))
        elif stripped.startswith('### '):
            rendered.append((plain_pdf_inline_text(stripped[4:].strip()), 'bold', 12))
        elif stripped.startswith('- '):
            rendered.append(('- ' + plain_pdf_inline_text(stripped[2:].strip()), 'regular', 10))
        else:
            rendered.append((plain_pdf_inline_text(stripped), 'regular', 10))
    return rendered


def write_plain_text_pdf(markdown_text: str, pdf_path: Path) -> str:
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    page_width, page_height = 612, 792
    margin_x, margin_y = 54, 54
    y_start = page_height - margin_y
    y_min = margin_y
    usable_width = page_width - 2 * margin_x
    pages: list[list[tuple[str, str, int, int]]] = [[]]
    y = y_start

    for text, font, size in markdown_lines_for_plain_pdf(markdown_text):
        line_height = max(size + 4, 13)
        max_chars = max(28, int(usable_width / (size * 0.52)))
        wrapped = textwrap.wrap(text, width=max_chars, replace_whitespace=False) if text else ['']
        for line in wrapped:
            if y < y_min:
                pages.append([])
                y = y_start
            pages[-1].append((line, font, size, y))
            y -= line_height
        if size >= 14:
            y -= 4

    objects: list[bytes] = []
    objects.append(b'<< /Type /Catalog /Pages 2 0 R >>')
    objects.append(b'')  # Filled after page objects are known.
    objects.append(b'<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica /Encoding /WinAnsiEncoding >>')
    objects.append(b'<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica-Bold /Encoding /WinAnsiEncoding >>')

    page_object_ids = []
    for page in pages:
        content_lines = []
        for line, font, size, y_pos in page:
            font_name = 'F2' if font == 'bold' else 'F1'
            content_lines.append(f'BT /{font_name} {size} Tf {margin_x} {y_pos} Td ({pdf_escape(line)}) Tj ET')
        content = '\n'.join(content_lines).encode('latin-1', errors='replace')
        content_id = len(objects) + 2
        page_id = len(objects) + 1
        page_object_ids.append(page_id)
        objects.append(
            f'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 {page_width} {page_height}] '
            f'/Resources << /Font << /F1 3 0 R /F2 4 0 R >> >> /Contents {content_id} 0 R >>'.encode('latin-1')
        )
        objects.append(b'<< /Length ' + str(len(content)).encode('ascii') + b' >>\nstream\n' + content + b'\nendstream')

    kids = ' '.join(f'{object_id} 0 R' for object_id in page_object_ids)
    objects[1] = f'<< /Type /Pages /Kids [{kids}] /Count {len(page_object_ids)} >>'.encode('latin-1')

    pdf = bytearray(b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n')
    offsets = [0]
    for index, obj in enumerate(objects, start=1):
        offsets.append(len(pdf))
        pdf.extend(f'{index} 0 obj\n'.encode('ascii'))
        pdf.extend(obj)
        pdf.extend(b'\nendobj\n')
    xref_offset = len(pdf)
    pdf.extend(f'xref\n0 {len(objects) + 1}\n'.encode('ascii'))
    pdf.extend(b'0000000000 65535 f \n')
    for offset in offsets[1:]:
        pdf.extend(f'{offset:010d} 00000 n \n'.encode('ascii'))
    pdf.extend(
        f'trailer\n<< /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref_offset}\n%%EOF\n'.encode('ascii')
    )
    pdf_path.write_bytes(bytes(pdf))
    return 'plain'


def export_investment_note_pdf(markdown_path: Path, pdf_path: Path, renderer: str = 'auto') -> tuple[Path, str, list[str]]:
    renderer = renderer.lower().strip()
    errors: list[str] = []
    if renderer in {'auto', 'weasyprint'}:
        try:
            renderer_used = export_pdf_with_weasyprint(markdown_path, pdf_path)
            return verify_pdf_file(pdf_path, renderer_used), renderer_used, errors
        except Exception as exc:
            errors.append(f'weasyprint: {exc}')
            if renderer == 'weasyprint':
                raise
    if renderer in {'auto', 'reportlab'}:
        try:
            renderer_used = export_pdf_with_reportlab(markdown_path, pdf_path)
            return verify_pdf_file(pdf_path, renderer_used), renderer_used, errors
        except Exception as exc:
            errors.append(f'reportlab: {exc}')
            if renderer == 'reportlab':
                raise
    if renderer in {'auto', 'pandoc'}:
        try:
            renderer_used = export_pdf_with_pandoc(markdown_path, pdf_path)
            return verify_pdf_file(pdf_path, renderer_used), renderer_used, errors
        except Exception as exc:
            errors.append(f'pandoc: {exc}')
            if renderer == 'pandoc':
                raise
    if renderer == 'plain' or (renderer == 'auto' and ALLOW_PLAIN_PDF_FALLBACK):
        markdown_path = verify_written_file(markdown_path, 'investment_note.md input')
        markdown_text = markdown_path.read_text(encoding='utf-8')
        renderer_used = write_plain_text_pdf(markdown_text, pdf_path)
        return verify_pdf_file(pdf_path, renderer_used), renderer_used, errors
    if renderer == 'auto':
        raise RuntimeError(
            'No rich PDF renderer succeeded and ALLOW_PLAIN_PDF_FALLBACK is False. Errors: '
            + '; '.join(errors)
        )
    raise ValueError("PDF_RENDERER must be 'auto', 'weasyprint', 'reportlab', 'pandoc', or 'plain'")


if EXPORT_INVESTMENT_NOTE_PDF:
    pdf_progress = NotebookProgress('Export PDF', 3, 'Preparing PDF export')
    pdf_progress.update(1, 'Checking rich PDF renderer packages')
    install_missing_notebook_packages(NOTEBOOK_RICH_PDF_PACKAGE_IMPORTS, "Rich PDF export")
    requested_pdf_renderer = PDF_RENDERER.lower().strip()
    rich_renderers = available_rich_pdf_renderers()
    if requested_pdf_renderer == 'auto' and not rich_renderers and not ALLOW_PLAIN_PDF_FALLBACK:
        environment = pdf_renderer_environment()
        raise RuntimeError(
            'No rich PDF renderer is available after the notebook package install step. '
            'The notebook will not write a plain fallback PDF because '
            'ALLOW_PLAIN_PDF_FALLBACK = False. '
            f'Renderer environment: {json.dumps(environment, indent=2)}'
        )
    pdf_progress.update(2, 'Rendering investment note PDF')
    investment_note_pdf_path, pdf_renderer_used, pdf_export_errors = export_investment_note_pdf(
        investment_note_path,
        ARTIFACT_DIR / PDF_FILENAME,
        requested_pdf_renderer,
    )
    pdf_size = investment_note_pdf_path.stat().st_size
    pdf_export_report = {
        'requested_renderer': PDF_RENDERER,
        'normalized_renderer': requested_pdf_renderer,
        'renderer_used': pdf_renderer_used,
        'pdf_path': str(investment_note_pdf_path),
        'pdf_size_bytes': pdf_size,
        'errors_before_success': pdf_export_errors,
        'renderer_environment': pdf_renderer_environment(),
    }
    pdf_export_report_path = write_json('pdf_export_report.json', pdf_export_report)
    pdf_progress.finish(f'PDF written with {pdf_renderer_used} ({format_bytes(pdf_size)})')
    fallback_note = ''
    if pdf_export_errors and pdf_renderer_used == 'plain':
        fallback_note = '\n\nUsed the plain PDF fallback because richer renderers were unavailable:\n' + '\n'.join(
            f'- {error}' for error in pdf_export_errors
        )
    display(Markdown(
        f'## PDF Export\n\nSaved PDF to `{investment_note_pdf_path}` using `{pdf_renderer_used}` '
        f'(`{format_bytes(pdf_size)}`). Export report: `{pdf_export_report_path}`.{fallback_note}'
    ))
else:
    investment_note_pdf_path = None
    display(Markdown('## PDF Export\n\nSkipped because `EXPORT_INVESTMENT_NOTE_PDF = False`.'))

In [ ]:
def artifact_inventory_markdown(directory: Path) -> str:
    directory = directory.resolve()
    rows = []
    if directory.exists():
        for path in sorted(directory.iterdir()):
            if path.is_file():
                rows.append(
                    f'| `{path.name}` | `{format_bytes(path.stat().st_size)}` | `{path}` |'
                )
    if not rows:
        return f'No files found in `{directory}`.'
    return '\n'.join([
        '| File | Size | Path |',
        '| --- | ---: | --- |',
        *rows,
    ])


required_artifacts = ['investment_note.md']
if EXPORT_INVESTMENT_NOTE_PDF:
    required_artifacts.append(PDF_FILENAME)

missing_artifacts = [
    name for name in required_artifacts
    if not (ARTIFACT_DIR / name).exists() or (ARTIFACT_DIR / name).stat().st_size <= 0
]
missing_note = ''
if missing_artifacts:
    missing_note = '\n\nMissing expected artifact(s): ' + ', '.join(f'`{name}`' for name in missing_artifacts)

display(Markdown(
    f'## Run Artifacts\n\nDirectory: `{ARTIFACT_DIR.resolve()}`\n\n'
    f'{artifact_inventory_markdown(ARTIFACT_DIR)}'
    f'{missing_note}'
))